# Soup Kernel Challenge: run on a Kaggle T4

Kaggle settings (right panel): **Accelerator: GPU T4 x2**, **Internet: on** (needs a
phone-verified Kaggle account). Put your repository in the first code cell, then **Run All**.
About 10 minutes: base Soup and your solution are trained and measured, both adapters are scored,
and the rule checks run.

Private repository? Add a Kaggle secret named `GITHUB_TOKEN` (Add-ons → Secrets) with a
read-only GitHub token; the notebook uses it to clone and never prints or saves it. An optional
secret `HF_TOKEN` avoids Hugging Face rate limits. Do not run your own GPU work in this notebook while the measured runs
go: they need the GPUs idle.

Every Run All clones your repository afresh: push to the default branch and leave `COMMIT`
empty (or set it to the new commit). For the submission: make this notebook public, set
`COMMIT` to your final commit, save a version with Save & Run All and put its link in your
README and `SUBMISSION.md`. The repository must be open to the jury by the deadline.

In [ ]:
REPO_URL = "https://github.com/<you>/<your-repo>.git"  # your repository
COMMIT = ""  # the commit you submit; empty = the default branch

In [ ]:
import os, subprocess
gpus = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                      capture_output=True, text=True).stdout
print(gpus)
assert "T4" in gpus, "Switch the accelerator to GPU T4 x2"
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass
!python -m pip install -q "soup-cli[train]==0.77.0" "transformers==5.16.1" "peft==0.20.0" "trl==0.29.1" "bitsandbytes==0.50.2" "accelerate==1.14.0" pytest

In [ ]:
import base64, os, shutil, subprocess
token = ""
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception as exc:
    print("No GITHUB_TOKEN secret, cloning without it:", type(exc).__name__)
auth = []
if token:  # the token goes in a header for this one command: never in a URL, a file or the output
    basic = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    auth = ["-c", f"http.extraHeader=Authorization: Basic {basic}"]
shutil.rmtree("/kaggle/working/repo", ignore_errors=True)  # always a fresh copy of your code
clone = subprocess.run(["git", *auth, "clone", "-q", REPO_URL, "/kaggle/working/repo"],
                       capture_output=True, text=True)
if clone.returncode:
    raise SystemExit("git clone failed: " + clone.stderr)
if COMMIT:
    subprocess.run(["git", "-C", "/kaggle/working/repo", "checkout", "-q", COMMIT], check=True)
print(subprocess.run(["git", "-C", "/kaggle/working/repo", "log", "-1", "--format=%H %s"],
                     capture_output=True, text=True).stdout)

In [ ]:
!cd /kaggle/working/repo && python task/prepare_data.py
!cd /kaggle/working/repo && python -m pytest solution -q

In [ ]:
!cd /kaggle/working/repo && python harness/run_task.py

In [ ]:
!cd /kaggle/working/repo && python harness/check_rules.py
!cd /kaggle/working/repo && python harness/score.py runs/result.json

While iterating, a quicker run of your solution alone (checks 1 and 5-8 need the base, so they fail
in this mode): `python harness/run_task.py --only solution --out runs_try`. Add `--profile` to get a
table of every CUDA kernel's GPU time in `runs/solution_profile/profile.json`. Logs of every run
are in `runs/<run>/train.log`.